# Retrieval Evaluation (LLM-as-Judge)

This notebook evaluates the book recommender's semantic search pipeline using an **LLM-as-judge** approach:

1. Run each test query through the retrieval pipeline
2. For each retrieved book, ask GPT-4o-mini: "Is this book relevant to the query?"
3. Compute standard IR metrics against the LLM's relevance judgments

This avoids the pool bias problem of narrow hand-curated test sets, where the system might return perfectly good books that simply weren't in our pre-selected list.

**Metrics:**
- **Precision@K** — Of the K books returned, what fraction did the judge deem relevant?
- **nDCG@K** — Are the most relevant results ranked highest?
- **Mean Relevance Score** — Average graded relevance (0-2) across results

In [8]:
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

load_dotenv()

books = pd.read_csv("books_with_emotions.csv")

CHROMA_PATH = "chroma_db"
db_books = Chroma(persist_directory=CHROMA_PATH, embedding_function=OpenAIEmbeddings())
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

print(f"Loaded Chroma with {db_books._collection.count()} documents")
print(f"Books CSV: {len(books)} rows")

Loaded Chroma with 5197 documents
Books CSV: 5197 rows


## 1. Test Queries

A diverse set of 20 queries spanning themes, genres, tones, and specifics. No pre-selected ISBNs needed — the LLM judge will evaluate whatever the retrieval system returns.

In [9]:
TEST_QUERIES = [
    "a story about forgiveness and redemption",
    "a murder mystery",
    "a science fiction space adventure",
    "a coming-of-age story",
    "a funny satirical novel",
    "a dark and suspenseful thriller",
    "a book about cooking and food",
    "a war novel set in World War II",
    "a philosophical exploration of life",
    "a fantasy novel with magic and dragons",
    "a detective solving crimes in a city",
    "a story about family secrets",
    "a journey of self-discovery",
    "a psychological thriller with twists",
    "a historical novel about ancient civilizations",
    "a love story set during a war",
    "a story about artificial intelligence and robots",
    "a book about nature and the wilderness",
    "exploring grief and loss",
    "a tale of forbidden love",
]

print(f"{len(TEST_QUERIES)} test queries")

20 test queries


## 2. Retrieve Results

In [10]:
K = 10

retrieval_results = {}

for query in TEST_QUERIES:
    recs = db_books.similarity_search(query, k=K)
    retrieved = []
    for rec in recs:
        try:
            isbn = int(rec.page_content.strip('"').split()[0])
            book_row = books[books["isbn13"] == isbn]
            if not book_row.empty:
                row = book_row.iloc[0]
                retrieved.append({
                    "isbn": isbn,
                    "title": row["title"],
                    "authors": row["authors"],
                    "description": str(row["description"])[:500],
                })
        except (ValueError, IndexError):
            continue
    retrieval_results[query] = retrieved

for q in TEST_QUERIES[:3]:
    print(f"\n'{q}' -> {len(retrieval_results[q])} books:")
    for b in retrieval_results[q][:3]:
        print(f"  - {b['title']} by {b['authors']}")

print(f"\n... ({len(TEST_QUERIES)} queries total)")


'a story about forgiveness and redemption' -> 10 books:
  - Go Tell it on the Mountain by James Baldwin
  - Forgiven by Karen Kingsbury
  - The Guardian by Jane Hamilton

'a murder mystery' -> 10 books:
  - The Murders in the Rue Morgue by Edgar Allan Poe
  - Mystery Play by Grant Morrison;Jon J. Muth
  - Farewell, My Lovely by Raymond Chandler

'a science fiction space adventure' -> 10 books:
  - The Starry Rift by James Tiptree
  - The Dig by Alan Dean Foster;Sean Clark
  - The Humanoids by Jack Williamson

... (20 queries total)


## 3. LLM Relevance Judging

For each (query, book) pair, the LLM assigns a **graded relevance score**:

| Score | Meaning |
|-------|---------|
| 2 | Highly relevant — the book clearly fits the query |
| 1 | Partially relevant — some thematic overlap but not a strong match |
| 0 | Not relevant — the book does not fit the query |

In [11]:
JUDGE_PROMPT = """You are an impartial relevance judge for a book recommendation system.

Given a user's search query and a book's metadata, rate how relevant the book is to the query.

Scoring:
- 2 = Highly relevant: the book clearly matches the query's intent (genre, theme, tone)
- 1 = Partially relevant: some overlap in theme or genre, but not a strong match
- 0 = Not relevant: the book does not fit the query

Respond with ONLY a JSON object: {"score": <0|1|2>, "reason": "<one sentence>"}
"""


def judge_relevance(query: str, book: dict) -> dict:
    user_msg = (
        f"Query: {query}\n\n"
        f"Book title: {book['title']}\n"
        f"Author: {book['authors']}\n"
        f"Description: {book['description']}"
    )
    response = llm.invoke([
        ("system", JUDGE_PROMPT),
        ("human", user_msg),
    ])
    try:
        result = json.loads(response.content)
        return {"score": int(result["score"]), "reason": result.get("reason", "")}
    except (json.JSONDecodeError, KeyError, ValueError):
        return {"score": 0, "reason": "parse error"}


# Test with one example
sample_query = TEST_QUERIES[0]
sample_book = retrieval_results[sample_query][0]
sample_judgment = judge_relevance(sample_query, sample_book)
print(f"Query: '{sample_query}'")
print(f"Book: '{sample_book['title']}'")
print(f"Judgment: {sample_judgment}")

Query: 'a story about forgiveness and redemption'
Book: 'Go Tell it on the Mountain'
Judgment: {'score': 1, 'reason': 'The book explores themes of guilt and spiritual strivings, which can relate to forgiveness and redemption, but it is not explicitly focused on those themes.'}


In [12]:
all_judgments = {}
total_pairs = sum(len(v) for v in retrieval_results.values())
judged = 0

for query in TEST_QUERIES:
    query_judgments = []
    for book in retrieval_results[query]:
        judgment = judge_relevance(query, book)
        query_judgments.append({
            "title": book["title"],
            "isbn": book["isbn"],
            "score": judgment["score"],
            "reason": judgment["reason"],
        })
        judged += 1
    all_judgments[query] = query_judgments
    
    relevant_count = sum(1 for j in query_judgments if j["score"] >= 1)
    highly_relevant = sum(1 for j in query_judgments if j["score"] == 2)
    print(f"'{query[:50]}...' -> {highly_relevant} highly relevant, {relevant_count} total relevant out of {len(query_judgments)}")

print(f"\nJudged {judged} query-book pairs total.")

'a story about forgiveness and redemption...' -> 3 highly relevant, 9 total relevant out of 10
'a murder mystery...' -> 9 highly relevant, 10 total relevant out of 10
'a science fiction space adventure...' -> 9 highly relevant, 10 total relevant out of 10
'a coming-of-age story...' -> 10 highly relevant, 10 total relevant out of 10
'a funny satirical novel...' -> 8 highly relevant, 10 total relevant out of 10
'a dark and suspenseful thriller...' -> 7 highly relevant, 10 total relevant out of 10
'a book about cooking and food...' -> 10 highly relevant, 10 total relevant out of 10
'a war novel set in World War II...' -> 5 highly relevant, 6 total relevant out of 10
'a philosophical exploration of life...' -> 9 highly relevant, 10 total relevant out of 10
'a fantasy novel with magic and dragons...' -> 7 highly relevant, 8 total relevant out of 10
'a detective solving crimes in a city...' -> 9 highly relevant, 10 total relevant out of 10
'a story about family secrets...' -> 7 highly releva

## 4. Compute Metrics

In [13]:
def precision_at_k(scores: list, k: int, threshold: int = 1) -> float:
    """Fraction of top-k results with score >= threshold."""
    top_k = scores[:k]
    if not top_k:
        return 0.0
    return sum(1 for s in top_k if s >= threshold) / k


def ndcg_at_k(scores: list, k: int) -> float:
    """nDCG@K using graded relevance scores (0, 1, 2)."""
    top_k = scores[:k]
    dcg = sum(s / np.log2(i + 2) for i, s in enumerate(top_k))
    ideal = sorted(scores, reverse=True)[:k]
    idcg = sum(s / np.log2(i + 2) for i, s in enumerate(ideal))
    if idcg == 0:
        return 0.0
    return dcg / idcg


def mean_relevance(scores: list) -> float:
    """Average relevance score (0-2 scale)."""
    if not scores:
        return 0.0
    return sum(scores) / len(scores)


K_VALUES = [3, 6, 10]

rows = []
for query in TEST_QUERIES:
    judgments = all_judgments[query]
    scores = [j["score"] for j in judgments]

    row = {"query": query}
    for k in K_VALUES:
        row[f"P@{k}"] = precision_at_k(scores, k)
        row[f"nDCG@{k}"] = ndcg_at_k(scores, k)
    row["mean_rel"] = mean_relevance(scores)
    rows.append(row)

results_df = pd.DataFrame(rows)
print("Metrics computed.")

Metrics computed.


## 5. Results

### Summary (averaged across all queries)

In [14]:
metric_cols = [c for c in results_df.columns if c != "query"]
summary = results_df[metric_cols].mean().to_frame("mean").T
summary.index = ["Average"]
summary.round(3)

,P@3,nDCG@3,P@6,nDCG@6,P@10,nDCG@10,mean_rel
Average,1.0,0.927,0.95,0.896,0.935,0.964,1.665


### Per-Query Breakdown

In [15]:
breakdown = results_df.copy()
breakdown = breakdown.sort_values("mean_rel", ascending=False)
breakdown.round(3)

,query,P@3,nDCG@3,P@6,nDCG@6,P@10,nDCG@10,mean_rel
3,a coming-of-age story,1.0,1.000,1.000,1.000,1.0,1.000,2.0
6,a book about cooking and food,1.0,1.000,1.000,1.000,1.0,1.000,2.0
10,a detective solving crimes in a city,1.0,1.000,1.000,0.946,1.0,0.992,1.9
1,a murder mystery,1.0,1.000,1.000,0.946,1.0,0.992,1.9
2,a science fiction space adventure,1.0,1.000,1.000,0.946,1.0,0.992,1.9
16,a story about artificial intelligence and robots,1.0,1.000,1.000,1.000,1.0,1.000,1.9
8,a philosophical exploration of life,1.0,0.852,1.000,0.905,1.0,0.961,1.9
18,exploring grief and loss,1.0,0.883,1.000,0.924,1.0,0.975,1.8
12,a journey of self-discovery,1.0,1.000,1.000,1.000,1.0,0.993,1.8
19,a tale of forbidden love,1.0,1.000,1.000,0.941,1.0,0.985,1.8


### Detailed Judgments

Inspect the LLM's reasoning for each query-book pair.

In [16]:
for query in TEST_QUERIES[:5]:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"{'='*70}")
    for j in all_judgments[query]:
        icon = {2: "++", 1: "+ ", 0: "--"}[j["score"]]
        print(f"  [{icon}] {j['title']} (score={j['score']})")
        print(f"       {j['reason']}")


Query: 'a story about forgiveness and redemption'
  [+ ] Go Tell it on the Mountain (score=1)
       The book explores themes of guilt and spiritual strivings, which relate to forgiveness and redemption, but it is not solely focused on those themes.
  [++] Forgiven (score=2)
       The book directly addresses themes of forgiveness and redemption through the protagonist's journey.
  [++] The Guardian (score=2)
       The book clearly matches the query's intent by focusing on themes of mercy and forgiveness.
  [--] The Great Gatsby (score=0)
       The Great Gatsby primarily focuses on themes of wealth, love, and the American Dream, rather than forgiveness and redemption.
  [+ ] Children of the Alley (score=1)
       The book touches on themes of family and moral struggles, but it does not explicitly focus on forgiveness and redemption.
  [+ ] The Trial of God (as it was Held on February 25, 1649, in Shamgorod) (score=1)
       The book touches on themes of faith and moral questioning, 

### Interpretation

This evaluation uses an **LLM-as-judge** approach, which avoids the pool bias problem of hand-curated test sets. Instead of pre-selecting a narrow list of "correct" books, we let the judge evaluate whatever the retrieval system actually returns.

**How to read the scores:**

- **Precision@K** — Fraction of top-K results judged as relevant (score >= 1). A P@6 of 0.8 means ~5 out of 6 results are at least partially relevant.
- **nDCG@K** — Measures ranking quality using graded scores (0/1/2). Values closer to 1.0 mean highly relevant books are ranked first.
- **Mean Relevance** — Average score across all retrieved books (scale 0-2). Above 1.0 means the average result is at least partially relevant.

**Strengths and limitations of LLM-as-judge:**
- (+) Scales to any number of queries without manual labeling
- (+) Provides graded relevance (not just binary) and explanations
- (+) Evaluates the actual system outputs, not a pre-selected subset
- (-) The judge LLM may have its own biases or make occasional errors
- (-) Costs API calls (one per query-book pair)